# Using Iterators in PythonLand — OOP Edition

**Theme:** financial market-data engineering and financial-news intelligence.

All 13 requested subtopics are covered in order. The examples use classes, methods, custom iterators, generators, and streaming pipelines rather than isolated toy examples.

## Learning Map

`Iterable → Custom Iterator → for loop → iterator functions → enumerate() → zip() → unzip → file streaming → generator methods → text extraction → streaming analytics`

## 1. Introduction to iterators

**Concept:** An iterator produces values sequentially while preserving its current position.

In [ ]:
class PriceSeries:
    def __init__(self, ticker, prices):
        self.ticker = ticker
        self.prices = prices

    def iterator(self):
        return iter(self.prices)

    def average_price(self):
        return sum(self.prices) / len(self.prices)


bhp = PriceSeries(
    "BHP",
    [41.70, 42.15, 41.92, 43.08, 42.76, 44.21]
)

prices = bhp.iterator()

print(bhp.ticker)
print("First:", next(prices))
print("Second:", next(prices))
print("Average:", round(bhp.average_price(), 2))

## 2. Iterators vs. Iterables

**Concept:** An iterable can create an iterator, while an iterator also stores traversal state and implements `__next__()`.


In [2]:
class Portfolio:
    def __init__(self, positions):
        self.positions = positions

    def __iter__(self):
        return iter(self.positions.items())


portfolio = Portfolio({
    "BHP": 0.18,
    "CBA": 0.27,
    "CSL": 0.20,
    "RIO": 0.20,
    "WES": 0.15
})

print("Iterable:", hasattr(portfolio, "__iter__"))
print("Iterator:", hasattr(portfolio, "__next__"))

portfolio_iterator = iter(portfolio)

print(next(portfolio_iterator))
print(next(portfolio_iterator))
print(list(portfolio_iterator))

Iterable: True
Iterator: False
('BHP', 0.18)
('CBA', 0.27)
[('CSL', 0.2), ('RIO', 0.2), ('WES', 0.15)]


## 3. Iterating over iterables (1)

**Concept:** A `for` loop automatically calls `__iter__()` and consumes the resulting iterator.

In [4]:
class TradeBook:
    def __init__(self, trades):
        self.trades = trades

    def __iter__(self):
        return iter(self.trades)

    def total_notional(self):
        return sum(
            trade["quantity"] * trade["price"]
            for trade in self
        )


trade_book = TradeBook([
    {"ticker": "BHP", "side": "BUY", "quantity": 2400, "price": 41.70},
    {"ticker": "CBA", "side": "SELL", "quantity": 800, "price": 178.25},
    {"ticker": "CSL", "side": "BUY", "quantity": 350, "price": 312.40},
    {"ticker": "RIO", "side": "BUY", "quantity": 1200, "price": 119.80},
])

for trade in trade_book:
    notional = trade["quantity"] * trade["price"]
    print(f'{trade["side"]:4} {trade["ticker"]:4} ${notional:,.2f}')

print(f"Total notional: ${trade_book.total_notional():,.2f}")

BUY  BHP  $100,080.00
SELL CBA  $142,600.00
BUY  CSL  $109,340.00
BUY  RIO  $143,760.00
Total notional: $495,780.00


## 4. Iterating over iterables (2)

**Concept:** An iterable class can expose business-level traversal over nested data without exposing its internal representation.

In [6]:
class MarketSnapshot:
    def __init__(self, ticker, metrics, venues):
        self.ticker = ticker
        self.metrics = metrics
        self.venues = venues

    def __iter__(self):
        for metric, value in self.metrics.items():
            yield metric, value

    def describe(self):
        for metric, value in self:
            print(f"{self.ticker} | {metric:<12} | {value}")


snapshot = MarketSnapshot(
    "BHP",
    {
        "price": 41.70,
        "volume": 1_842_000,
        "spread_bps": 3.7,
        "volatility": 0.284
    },
    ("ASX", "Chi-X")
)

snapshot.describe()

for venue in snapshot.venues:
    print("Venue:", venue)

BHP | price        | 41.7
BHP | volume       | 1842000
BHP | spread_bps   | 3.7
BHP | volatility   | 0.284
Venue: ASX
Venue: Chi-X


## 5. Iterators as function arguments

**Concept:** An iterator can be passed into analytical functions so data access and calculation remain separate concerns.

In [ ]:
class MarketSeries:
    def __init__(self, ticker, prices, volumes):
        self.ticker = ticker
        self.prices = prices
        self.volumes = volumes

    def price_iterator(self):
        return iter(self.prices)

    def volume_iterator(self):
        return iter(self.volumes)


def calculate_vwap(price_iterator, volume_iterator):
    weighted_value = 0
    total_volume = 0

    for price, volume in zip(price_iterator, volume_iterator):
        weighted_value += price * volume
        total_volume += volume

    return weighted_value / total_volume


series = MarketSeries(
    "BHP",
    [41.70, 42.15, 41.92, 43.08],
    [1_842_000, 2_130_000, 1_540_000, 2_450_000]
)

vwap = calculate_vwap(
    series.price_iterator(),
    series.volume_iterator()
)

print(f"{series.ticker} VWAP: ${vwap:.4f}")

## 6. Playing with iterators

**Concept:** A custom iterator can explicitly manage state and stop with `StopIteration` when its data is exhausted.

In [ ]:
class SignalIterator:
    def __init__(self, signals):
        self.signals = signals
        self.position = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.position >= len(self.signals):
            raise StopIteration

        signal = self.signals[self.position]
        self.position += 1
        return signal


signals = SignalIterator([
    {"ticker": "BHP", "signal": "BUY", "confidence": 0.82},
    {"ticker": "CBA", "signal": "HOLD", "confidence": 0.64},
    {"ticker": "CSL", "signal": "BUY", "confidence": 0.77},
    {"ticker": "RIO", "signal": "SELL", "confidence": 0.71},
])

for _ in range(5):
    signal = next(signals, None)
    if signal is None:
        print("Signal stream exhausted")
        break
    print(signal)

## 7. Using `enumerate()`

**Concept:** `enumerate()` attaches a position to each item without requiring a manual counter.

In [ ]:
class MomentumScanner:
    def __init__(self, observations):
        self.observations = observations

    def ranked(self):
        return sorted(
            self.observations,
            key=lambda row: row["momentum"],
            reverse=True
        )

    def report(self):
        for rank, row in enumerate(self.ranked(), start=1):
            bucket = (
                "HIGH" if row["momentum"] > 0.07
                else "MEDIUM" if row["momentum"] > 0.05
                else "LOW"
            )
            print(
                f"{rank}. {row['ticker']:<4} "
                f"momentum={row['momentum']:>6.2%} "
                f"bucket={bucket}"
            )

scanner = MomentumScanner([
    {"ticker": "BHP", "momentum": 0.084},
    {"ticker": "CBA", "momentum": 0.071},
    {"ticker": "CSL", "momentum": 0.063},
    {"ticker": "RIO", "momentum": 0.058},
    {"ticker": "WES", "momentum": 0.044},
])

scanner.report()

1. BHP  momentum= 8.40% bucket=HIGH
2. CBA  momentum= 7.10% bucket=HIGH
3. CSL  momentum= 6.30% bucket=MEDIUM
4. RIO  momentum= 5.80% bucket=MEDIUM
5. WES  momentum= 4.40% bucket=LOW


## 8. Using `zip()`

**Concept:** `zip()` aligns multiple iterables into logical records without manual index management.

In [3]:
class MarketSession:
    def __init__(self, tickers, prices, returns, volumes):
        self.tickers = tickers
        self.prices = prices
        self.returns = returns
        self.volumes = volumes

    def records(self):
        return zip(
            self.tickers,
            self.prices,
            self.returns,
            self.volumes
        )

    def report(self):
        for ticker, price, daily_return, volume in self.records():
            notional = price * volume
            print(
                f"{ticker:<4} "
                f"${price:>7.2f} "
                f"{daily_return:>7.2%} "
                f"${notional:>12,.0f}"
            )


session = MarketSession(
    ["BHP", "CBA", "CSL", "RIO", "WES"],
    [41.70, 178.25, 312.40, 119.80, 102.15],
    [0.021, -0.008, 0.014, 0.006, 0.011],
    [1_842_000, 932_100, 521_800, 1_205_000, 740_300]
)

session.report()

BHP  $  41.70   2.10% $  76,811,400
CBA  $ 178.25  -0.80% $ 166,146,825
CSL  $ 312.40   1.40% $ 163,010,320
RIO  $ 119.80   0.60% $ 144,359,000
WES  $ 102.15   1.10% $  75,621,645


## 9. Using `*` and `zip()` to 'unzip'

**Concept:** `zip(*records)` transposes row-oriented records into column-oriented iterables.

In [4]:
class MarketTable:
    def __init__(self, records):
        self.records = records

    def columns(self):
        return zip(*self.records)

    def statistics(self):
        tickers, prices, returns, volumes = self.columns()
        return {
            "tickers": tickers,
            "prices": prices,
            "returns": returns,
            "volumes": volumes,
            "average_return": sum(returns) / len(returns),
            "total_volume": sum(volumes)
        }


table = MarketTable([
    ("BHP", 41.70, 0.021, 1_842_000),
    ("CBA", 178.25, -0.008, 932_100),
    ("CSL", 312.40, 0.014, 521_800),
    ("RIO", 119.80, 0.006, 1_205_000),
    ("WES", 102.15, 0.011, 740_300),
])

stats = table.statistics()

print("Tickers:", stats["tickers"])
print("Prices:", stats["prices"])
print("Returns:", stats["returns"])
print(f"Average return: {stats['average_return']:.2%}")
print(f"Total volume: {stats['total_volume']:,}")

Tickers: ('BHP', 'CBA', 'CSL', 'RIO', 'WES')
Prices: (41.7, 178.25, 312.4, 119.8, 102.15)
Returns: (0.021, -0.008, 0.014, 0.006, 0.011)
Average return: 0.88%
Total volume: 5,241,200


## 10. Using iterators to load large files into memory

**Concept:** A file-streaming class can yield one parsed record at a time instead of loading an entire CSV into memory.

In [6]:
from pathlib import Path


class MarketFileStreamer:
    def __init__(self, path):
        self.path = Path(path)

    def __iter__(self):
        with self.path.open() as file:
            next(file)

            for line in file:
                timestamp, ticker, price, volume = line.strip().split(",")

                yield {
                    "timestamp": timestamp,
                    "ticker": ticker,
                    "price": float(price),
                    "volume": int(volume)
                }


path = Path("market_ticks.csv")

path.write_text("\n".join([
    "timestamp,ticker,price,volume",
    "09:30:01,BHP,41.70,182400",
    "09:30:02,BHP,41.72,245100",
    "09:30:03,CBA,178.25,91200",
    "09:30:04,BHP,41.68,301500",
    "09:30:05,CSL,312.40,78300",
    "09:30:06,RIO,119.80,190400",
    "09:30:07,CBA,178.31,142700",
    "09:30:08,BHP,41.75,287900",
]))

volume_by_ticker = {}

for record in MarketFileStreamer(path):
    ticker = record["ticker"]
    volume_by_ticker[ticker] = (
        volume_by_ticker.get(ticker, 0) + record["volume"]
    )

print(volume_by_ticker)

{'BHP': 1016900, 'CBA': 233900, 'CSL': 78300, 'RIO': 190400}


## 11. Processing large amounts of Twitter data

**Concept:** Generator methods inside a class create lazy filters and transformations so only records being consumed are processed.

In [7]:
class FinancialNewsStream:
    def __init__(self, posts):
        self.posts = posts

    def relevant_posts(self, keywords):
        for post in self.posts:
            text = post["text"].lower()
            if any(keyword in text for keyword in keywords):
                yield post

    def engagement_stream(self, keywords):
        for post in self.relevant_posts(keywords):
            yield {
                **post,
                "engagement_score": (
                    post["likes"] * 1.5
                    + post["replies"] * 2
                    + post["reposts"] * 3
                )
            }


posts = [
    {"id": 101, "text": "BHP shares rally as iron ore prices recover",
     "likes": 420, "replies": 35, "reposts": 62},
    {"id": 102, "text": "CBA reports stronger quarterly profit growth",
     "likes": 810, "replies": 72, "reposts": 91},
    {"id": 103, "text": "NVDA demand remains strong across AI infrastructure",
     "likes": 1250, "replies": 140, "reposts": 210},
    {"id": 104, "text": "Markets fall as inflation expectations rise",
     "likes": 670, "replies": 88, "reposts": 102},
]

stream = FinancialNewsStream(posts)

for post in stream.engagement_stream(
    ["bhp", "cba", "nvda", "market"]
):
    print(
        f'#{post["id"]} '
        f'engagement={post["engagement_score"]:.0f}'
    )

#101 engagement=886
#102 engagement=1632
#103 engagement=2785
#104 engagement=1487


## 12. Extracting information for large amounts of Twitter data

**Concept:** A parser class can convert raw financial text into structured fields while a generator controls when each post is parsed.

In [8]:
import re


class FinancialPostParser:
    TICKER_PATTERN = re.compile(r"\$([A-Z]{2,5})")
    MOVE_PATTERN = re.compile(r"(-?\d+(?:\.\d+)?)%")

    def extract_ticker(self, text):
        match = self.TICKER_PATTERN.search(text)
        return match.group(1) if match else None

    def extract_move(self, text):
        match = self.MOVE_PATTERN.search(text)
        return float(match.group(1)) if match else None

    def sentiment(self, text):
        text = text.lower()
        positive = ["rallies", "gains", "jumps", "rises", "stronger", "positive"]
        negative = ["falls", "weaker", "decline", "drops", "concerns"]

        if any(word in text for word in positive):
            return "positive"
        if any(word in text for word in negative):
            return "negative"
        return "neutral"

    def parse(self, post):
        return {
            "ticker": self.extract_ticker(post),
            "move_pct": self.extract_move(post),
            "sentiment": self.sentiment(post),
            "text": post
        }


posts = [
    "BHP $BHP rallies 4.2% after iron ore prices rise",
    "CBA $CBA gains 1.8% as profit beats expectations",
    "NVDA $NVDA jumps 5.1% on stronger AI demand",
    "RIO $RIO falls 2.3% after weaker commodity prices",
    "CSL $CSL rises 0.9% following positive guidance",
]

parser = FinancialPostParser()

for record in (parser.parse(post) for post in iter(posts)):
    print(record)

{'ticker': 'BHP', 'move_pct': 4.2, 'sentiment': 'positive', 'text': 'BHP $BHP rallies 4.2% after iron ore prices rise'}
{'ticker': 'CBA', 'move_pct': 1.8, 'sentiment': 'positive', 'text': 'CBA $CBA gains 1.8% as profit beats expectations'}
{'ticker': 'NVDA', 'move_pct': 5.1, 'sentiment': 'positive', 'text': 'NVDA $NVDA jumps 5.1% on stronger AI demand'}
{'ticker': 'RIO', 'move_pct': 2.3, 'sentiment': 'negative', 'text': 'RIO $RIO falls 2.3% after weaker commodity prices'}
{'ticker': 'CSL', 'move_pct': 0.9, 'sentiment': 'positive', 'text': 'CSL $CSL rises 0.9% following positive guidance'}


## 13. Extracting information for large amounts of Twitter data

**Concept:** Multiple lazy stages can be chained into a reusable OOP pipeline that filters, parses, transforms, and aggregates records as they stream through.

In [10]:
import re
from collections import defaultdict


class MarketPostPipeline:
    def __init__(self, posts):
        self.posts = posts

    def filter_market_posts(self):
        keywords = {"bhp", "cba", "rio", "csl", "nvda"}

        for post in self.posts:
            if any(
                keyword in post["text"].lower()
                for keyword in keywords
            ):
                yield post

    def extract_ticker(self, text):
        match = re.search(r"\$([A-Z]{2,5})", text)
        return match.group(1) if match else None

    def extract_move(self, text):
        match = re.search(r"(-?\d+(?:\.\d+)?)%", text)
        return float(match.group(1)) if match else 0.0

    def classify(self, move):
        if move > 0:
            return "positive"
        if move < 0:
            return "negative"
        return "neutral"

    def parsed_stream(self):
        for post in self.filter_market_posts():
            move = self.extract_move(post["text"])

            yield {
                "ticker": self.extract_ticker(post["text"]),
                "move_pct": move,
                "sentiment": self.classify(move),
                "likes": post["likes"],
                "text": post["text"]
            }

    def aggregate(self):
        summary = defaultdict(
            lambda: {
                "posts": 0,
                "positive": 0,
                "negative": 0,
                "likes": 0,
                "absolute_move": 0.0
            }
        )

        for record in self.parsed_stream():
            ticker = record["ticker"]
            summary[ticker]["posts"] += 1
            summary[ticker][record["sentiment"]] += 1
            summary[ticker]["likes"] += record["likes"]
            summary[ticker]["absolute_move"] += abs(record["move_pct"])

        return summary


posts = [
    {"text": "BHP $BHP rallies 4.2% after iron ore prices rise", "likes": 420},
    {"text": "CBA $CBA gains 1.8% as profit beats expectations", "likes": 810},
    {"text": "NVDA $NVDA jumps 5.1% on stronger AI demand", "likes": 1250},
    {"text": "RIO $RIO falls 2.3% after weaker commodity prices", "likes": 390},
    {"text": "CSL $CSL rises 0.9% following positive guidance", "likes": 510},
    {"text": "BHP $BHP gains 2.1% after commodity outlook improves", "likes": 620},
    {"text": "CBA $CBA falls 0.7% as credit concerns emerge", "likes": 730},
    {"text": "BHP $BHP gains 3.4% as copper demand improves", "likes": 910},
]

pipeline = MarketPostPipeline(posts)

for ticker, stats in pipeline.aggregate().items():
    average_move = stats["absolute_move"] / stats["posts"]

    print(
        f"{ticker:<5} "
        f"posts={stats['posts']} "
        f"positive={stats['positive']} "
        f"negative={stats['negative']} "
        f"likes={stats['likes']:,} "
        f"avg_abs_move={average_move:.2f}%"
    )

BHP   posts=3 positive=3 negative=0 likes=1,950 avg_abs_move=3.23%
CBA   posts=2 positive=2 negative=0 likes=1,540 avg_abs_move=1.25%
NVDA  posts=1 positive=1 negative=0 likes=1,250 avg_abs_move=5.10%
RIO   posts=1 positive=1 negative=0 likes=390 avg_abs_move=2.30%
CSL   posts=1 positive=1 negative=0 likes=510 avg_abs_move=0.90%
